This R program loads the Arrow data file and extracts a selection of columns from the Los Angeles area of responsibility, removing any arrest that appears to be from another state. 

It converts the date-time of the arrest to a simple date, then exports to a new Google Sheet.

In [ ]:

pacman::p_load("tidyverse", "lubridate", "arrow", "duckdb", "googlesheets4", "janitor", "glue")
print (getwd())
bln_rawdata <- read_feather("bln_arrest_recipe_data.arrow")
la_rawdata <-
  bln_rawdata |>
  # ugh. Typo in the column name. Fix another time.
  rename ( bln_charge_group = bln_charge_grouop) |>
  filter(
    str_detect(apprehension_aor, "Los Angeles") &
      bln_arrest_state == "CALIFORNIA"
  ) |>
    mutate(arrest_date = as_date(apprehension_date)) |>
    select(
      orig_row_number = arrest_id,
      arrest_date,
      apprehension_site_landmark,
      arrest_street_custodial = apprehension_method_recoded,
      arrest_criminality = apprehension_criminality,
      bln_arrest_charge_code,
      bln_charge,
      bln_charge_group_code,
      bln_charge_group,
      bln_charge_special,
      detention_facility,
      detention_state,
      detention_city,
      detention_county,
      detainer_facility,
      detainer_county,
      birth_year,
      citizenship_country,
      case_status_recoded,
      departed_date,
      departure_country,
      gender,
      is_last_arrest,
      bln_person_id
    )


[1] "/Users/sarah/Github/deportation-data/data"


Export to Google sheets: Create the spreadsheet if it doesn't exist, and delete the specific page and recreate it if it does. This seems to take forever. It would probably be easier to export to Excel then import to GSheets.

In [17]:

sheet_name <- "ice-arrests-la-extract"
wksht_name <- "raw data"
found_sheet <- gs4_find (sheet_name)

if (nrow(found_sheet) == 0 ) {
  ss = gs4_create(sheet_name)
} else {
  ss <- found_sheet$id[1]
  existing_sheets <- sheet_names(ss)
  if ( wksht_name %in% existing_sheets) {
    sheet_delete (ss, sheet=wksht_name)
  }
}

sheet_write (la_rawdata, ss, sheet= wksht_name)




Auto-refreshing stale OAuth token.
Auto-refreshing stale OAuth token.
✔ Deleting 1 sheet from ice-arrests-la-extract:
• raw data
✔ Writing to ice-arrests-la-extract.
✔ Writing to sheet raw data.



Export to a csv just so that we can show an import into GOogle sheets


In [18]:
la_rawdata |>
  write_csv( file="la_arrests.csv", na="")
